In [0]:
%pip install elasticsearch==8.19.0
%restart_python

# Delete stale locations from Elasticsearch (oxjob #850)

The locations index sync (`sync_locations`) only upserts, and `locations_mapped`
is a nightly full rebuild — a deleted location just stops appearing, so its ES
doc lived in the index forever (the ~25M-ghost problem that made locations-v1
untrustworthy). This notebook is the delete path, mirroring `delete_works`
(#784): it consumes the ledger `openalex.works.deleted_locations` (written by
`notebooks/end2end/TrackDeletedLocations`) and bulk-deletes every entry not yet
stamped `es_deleted_at`. The ledger `id` IS the ES `_id`
(`native_id_namespace:native_id`) — no prefixing.

- Runs after `Sync_Locations_to_Elasticsearch` in end2end. Deletes are
  404-tolerant (already-absent docs count as done), so retries are safe.
- Rows are stamped `es_deleted_at` only after the delete pass; non-404 failures
  stay unstamped and are retried next night.
- **Guard**: aborts if the eligible set in `locations_mapped` is empty or the
  pending set exceeds `guard_fraction` (default 0.5%) of the live index doc
  count, unless `guard_override=true` (job parameter
  `deleted_locations_guard_override`).
- **`is_full_reconcile=true`** (manual, run-now-with-parameters): scrolls every
  `_id` out of the index and ledgers any doc with no eligible row in
  `locations_mapped` — catches ghosts that predate the ledger, including
  deletions that landed between the 08-30 locations-v3 full sync and the first
  TrackDeletedLocations run. Slow (full index scroll); intended as a one-off
  after this ships.


In [0]:
import logging
from pyspark.sql import functions as F
from elasticsearch import Elasticsearch, helpers

logging.basicConfig(level=logging.WARNING, format="[%(asctime)s]: %(message)s")
log = logging.getLogger(__name__)

# Must match CONFIG["index_name"] in notebooks/elastic/sync_locations.
ELASTIC_INDEX = "locations-v3"
ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")

dbutils.widgets.text("guard_override", "false")
dbutils.widgets.text("guard_fraction", "0.005")
dbutils.widgets.text("is_full_reconcile", "false")
dbutils.widgets.text("env_suffix", "")

GUARD_OVERRIDE = dbutils.widgets.get("guard_override").lower() == "true"
GUARD_FRACTION = float(dbutils.widgets.get("guard_fraction"))
IS_FULL_RECONCILE = dbutils.widgets.get("is_full_reconcile").lower() == "true"
ENV_SUFFIX = dbutils.widgets.get("env_suffix")

CATALOG = f"openalex{ENV_SUFFIX}"
MAPPED = f"{CATALOG}.works.locations_mapped"
LEDGER = f"{CATALOG}.works.deleted_locations"
ES_SCAN_SCRATCH = f"{CATALOG}.works._tmp_es_location_ids_850"

# Same eligibility filter as sync_locations / TrackDeletedLocations — keep byte-identical.
ELIGIBLE_SQL = f"""
SELECT DISTINCT CONCAT(native_id_namespace, ':', native_id) AS id
FROM {MAPPED}
WHERE native_id IS NOT NULL AND native_id_namespace IS NOT NULL
  AND work_id IS NOT NULL AND work_id > 0
"""

client = Elasticsearch(
    hosts=[ELASTIC_URL],
    request_timeout=180,
    max_retries=5,
    retry_on_timeout=True,
)

print(f"guard_override: {GUARD_OVERRIDE}, guard_fraction: {GUARD_FRACTION}, full_reconcile: {IS_FULL_RECONCILE}")


### Optional: full ES-scan reconcile (manual backfill of pre-ledger ghosts)

In [0]:
if IS_FULL_RECONCILE:
    print(f"FULL RECONCILE: scrolling all _ids from {ELASTIC_INDEX} into {ES_SCAN_SCRATCH}...", flush=True)
    spark.sql(f"DROP TABLE IF EXISTS {ES_SCAN_SCRATCH}")
    spark.sql(f"CREATE TABLE {ES_SCAN_SCRATCH} (id STRING) USING DELTA")

    batch, total = [], 0
    def flush(rows):
        if rows:
            spark.createDataFrame([(r,) for r in rows], "id STRING") \
                .write.mode("append").saveAsTable(ES_SCAN_SCRATCH)

    for hit in helpers.scan(
        client, index=ELASTIC_INDEX, query={"query": {"match_all": {}}},
        _source=False, size=10_000, scroll="15m",
    ):
        batch.append(hit["_id"])
        if len(batch) >= 1_000_000:
            flush(batch)
            total += len(batch)
            batch = []
            print(f"  ...scanned {total:,}", flush=True)
    flush(batch)
    total += len(batch)
    print(f"Scanned {total:,} ES ids.", flush=True)

    ghosts = spark.sql(f"""
        INSERT INTO {LEDGER}
        SELECT e.id, current_date(), current_timestamp(), NULL
        FROM {ES_SCAN_SCRATCH} e
        LEFT ANTI JOIN ({ELIGIBLE_SQL}) m ON e.id = m.id
        LEFT ANTI JOIN {LEDGER} l ON e.id = l.id
    """).collect()[0].num_inserted_rows
    print(f"Full reconcile ledgered {ghosts:,} ghost docs.")
else:
    print("Incremental run (ledger-driven only).")


### Delete pass

In [0]:
pending_df = spark.sql(f"SELECT id FROM {LEDGER} WHERE es_deleted_at IS NULL")
pending_count = pending_df.count()
eligible_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {MAPPED}
    WHERE native_id IS NOT NULL AND native_id_namespace IS NOT NULL
      AND work_id IS NOT NULL AND work_id > 0
""").collect()[0].cnt
live_count = client.count(index=ELASTIC_INDEX)["count"]
print(f"Pending deletes: {pending_count:,}; live index docs: {live_count:,}; eligible rows in {MAPPED}: {eligible_count:,}")

if pending_count > 0:
    if eligible_count == 0:
        raise Exception(f"ABORT: no eligible rows in {MAPPED}; refusing to trust the ledger. No deletes issued.")
    if pending_count > GUARD_FRACTION * live_count and not GUARD_OVERRIDE:
        raise Exception(
            f"ABORT: {pending_count:,} pending deletes (> {GUARD_FRACTION:.2%} of {live_count:,} live docs). "
            "No deletes issued; ledger unchanged. If this is a sanctioned mass deletion, "
            "re-run with deleted_locations_guard_override=true."
        )

    ids = [r.id for r in pending_df.collect()]

    def actions():
        for doc_id in ids:
            yield {"_op_type": "delete", "_index": ELASTIC_INDEX, "_id": doc_id}

    deleted = missing = 0
    failed_ids = []
    for success, info in helpers.parallel_bulk(
        client, actions(), chunk_size=700, thread_count=4,
        raise_on_error=False, raise_on_exception=False,
    ):
        item = info.get("delete", {})
        if success:
            deleted += 1
            if deleted % 100_000 == 0:
                print(f"  ...deleted {deleted:,} / {len(ids):,}", flush=True)
        elif item.get("status") == 404:
            missing += 1
        else:
            failed_ids.append(str(item.get("_id", "")))
            if len(failed_ids) <= 10:
                print(f"  DELETE failed: {info}", flush=True)

    print(f"Deleted {deleted:,}, already absent {missing:,}, failed {len(failed_ids):,} of {len(ids):,}.")

    if failed_ids:
        spark.createDataFrame([(i,) for i in failed_ids], "id STRING") \
            .createOrReplaceTempView("failed_deletes")
        spark.sql(f"""
            UPDATE {LEDGER} SET es_deleted_at = current_timestamp()
            WHERE es_deleted_at IS NULL
              AND id NOT IN (SELECT id FROM failed_deletes)
        """)
        print(f"Stamped all but {len(failed_ids):,} failures (left pending for retry).")
    else:
        spark.sql(f"UPDATE {LEDGER} SET es_deleted_at = current_timestamp() WHERE es_deleted_at IS NULL")
        print("Stamped es_deleted_at on all pending rows.")
else:
    print("Nothing to delete.")

spark.sql(f"DROP TABLE IF EXISTS {ES_SCAN_SCRATCH}")
client.indices.refresh(index=ELASTIC_INDEX)
print(f"{client.count(index=ELASTIC_INDEX)['count']:,} documents remain in {ELASTIC_INDEX}.")
client.close()
